# 003 · Sockets

> **Standard Library › Networking** · Python 3.14

The `socket` module is a thin wrapper around the operating system's BSD socket API — the layer under HTTP, database drivers and everything else on the network. It corresponds to `System.Net.Sockets.Socket`, with the convenience of `TcpListener`/`TcpClient` provided by a few helper functions. You rarely need raw sockets, but when you write a custom protocol, a test double or debug network code, you need to know how they behave.

**In this notebook you will learn:**

- how `socket` maps onto `Socket`, `TcpListener`, `TcpClient` and `UdpClient`
- a TCP echo server and client: `bind`, `listen`, `accept`, `connect`, `sendall`, `recv`
- using sockets with the `with` statement
- the message-framing gotcha (`recv` may return partial data) and a length-prefix fix with `struct`
- timeouts
- UDP in brief, and the `socketserver` module

## .NET mental model

| .NET | Python `socket` |
|---|---|
| `new Socket(AddressFamily.InterNetwork, SocketType.Stream, ProtocolType.Tcp)` | `socket.socket(socket.AF_INET, socket.SOCK_STREAM)` (the defaults) |
| `TcpListener(IPAddress.Loopback, 0)` + `Start()` | `socket.create_server(("127.0.0.1", 0))` or `bind` + `listen` |
| `listener.AcceptTcpClient()` / `socket.Accept()` | `conn, addr = server.accept()` |
| `new TcpClient(host, port)` / `socket.Connect(ep)` | `socket.create_connection((host, port))` |
| `stream.Write(bytes)` | `sock.sendall(data)` |
| `stream.Read(buffer)` | `sock.recv(bufsize)` → `bytes` (`b""` = closed) |
| `ReceiveTimeout` / `SendTimeout` | `sock.settimeout(seconds)` |
| `UdpClient.Send` / `Receive` | `SOCK_DGRAM` + `sendto` / `recvfrom` |
| `BinaryPrimitives.WriteInt32BigEndian` | `struct.pack("!I", n)` |
| `using var s = ...` | `with socket.socket() as s:` |

Addresses are tuples: `(host, port)` for IPv4. Port `0` means "let the OS choose a free port" — we use it throughout so the notebook never collides with anything running on your machine, and everything stays on `127.0.0.1` (offline).

## A TCP echo server

```csharp
// C#
var listener = new TcpListener(IPAddress.Loopback, 0);
listener.Start();
int port = ((IPEndPoint)listener.LocalEndpoint).Port;
using TcpClient client = listener.AcceptTcpClient();
NetworkStream stream = client.GetStream();
```

The raw sequence is `socket()` → `bind()` → `listen()` → `accept()` in a loop. `accept()` blocks until a client connects and returns a **new** socket for that connection plus the client's address. We serve each connection on its own thread and run the accept loop in a background daemon thread.

In [ ]:
import socket
import threading

def handle_client(conn):
    with conn:                                  # closes the connection socket at the end
        while data := conn.recv(1024):          # b"" means the client closed its side
            conn.sendall(data.upper())

def serve(listener, stop):
    listener.settimeout(0.1)                    # wake up regularly to check the stop flag
    while not stop.is_set():
        try:
            conn, addr = listener.accept()
        except TimeoutError:
            continue
        conn.settimeout(None)                   # back to blocking mode for the connection
        threading.Thread(target=handle_client, args=(conn,), daemon=True).start()

listener = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
listener.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
listener.bind(("127.0.0.1", 0))                 # port 0 → OS picks a free port
listener.listen()
host, port = listener.getsockname()
stop = threading.Event()
threading.Thread(target=serve, args=(listener, stop), daemon=True).start()
print("echo server on", host, "port assigned:", port > 0)

**Tip:** `socket.create_server(("127.0.0.1", 0))` does `socket` + `SO_REUSEADDR` + `bind` + `listen` in one call — the equivalent of `new TcpListener(...).Start()`. We spelled out the steps above so you can see them.

## A TCP client

```csharp
// C#
using var client = new TcpClient();
client.Connect("127.0.0.1", port);
var stream = client.GetStream();
stream.Write("hello"u8);
var buffer = new byte[1024];
int n = stream.Read(buffer);
```

`socket.create_connection((host, port), timeout)` is the `TcpClient` shortcut: it resolves the host, connects and returns a socket. Sockets send and receive **bytes**, never `str` — encode and decode explicitly *(see Standard Library › Text Processing › 004 · Unicode, Bytes and Encodings)*.

In [ ]:
with socket.create_connection(("127.0.0.1", port), timeout=5) as client:
    client.sendall("hello, sockets".encode("utf-8"))
    reply = client.recv(1024)
    print(reply)
    print(reply.decode("utf-8"))

- `sendall(data)` keeps sending until every byte is written. Plain `send()` may write only *part* of the data and returns how many bytes it sent — prefer `sendall`.
- `recv(n)` returns **up to** `n` bytes — whatever has arrived. It returns `b""` when the peer has closed the connection.
- `with socket... as s:` closes the socket on exit, like `using`.

Several clients at once — each connection gets its own server thread. We collect the replies and print them sorted:

In [ ]:
replies = []
lock = threading.Lock()

def client_job(name):
    with socket.create_connection(("127.0.0.1", port), timeout=5) as c:
        c.sendall(name.encode())
        answer = c.recv(1024).decode()
    with lock:
        replies.append(answer)

jobs = [threading.Thread(target=client_job, args=(n,)) for n in ["alice", "bob", "carol"]]
for j in jobs: j.start()
for j in jobs: j.join()
print(sorted(replies))

## ⚠️ The framing gotcha: `recv` may return partial data

TCP is a **byte stream**, not a message protocol. One `sendall` on one side does not mean one `recv` on the other: the data may arrive split into several pieces, or several messages may arrive glued together. The same is true for `NetworkStream.Read` in .NET — a single `Read` can return fewer bytes than were written.

To make this visible deterministically, here is a server that sends one logical message in two pieces with a short pause in between:

In [ ]:
import time

def slow_sender(listener_sock):
    conn, _ = listener_sock.accept()
    with conn:
        conn.sendall(b"first half / ")
        time.sleep(0.1)                         # simulate network delay
        conn.sendall(b"second half")

with socket.create_server(("127.0.0.1", 0)) as srv:
    t = threading.Thread(target=slow_sender, args=(srv,))
    t.start()
    with socket.create_connection(srv.getsockname(), timeout=5) as c:
        print("single recv got:", c.recv(1024))     # only what has arrived so far!
        rest = b""
        while chunk := c.recv(1024):                # keep reading until the peer closes
            rest += chunk
        print("the rest:        ", rest)
    t.join()

"Read until the peer closes" works for one message per connection, but not for a conversation. Real protocols define **framing**:

- a **delimiter** (e.g. `\n` — then use `sock.makefile("rb").readline()`), like HTTP headers or line-based protocols
- a **length prefix**: send the message size first, then exactly that many bytes

### Length-prefix framing with `struct`

`struct.pack("!I", n)` encodes `n` as a 4-byte unsigned int in network byte order (big-endian) — like `BinaryPrimitives.WriteUInt32BigEndian`. The reader first reads exactly 4 bytes, unpacks the length, then reads exactly that many bytes, looping over `recv` until it has them all.

In [ ]:
import struct

HEADER = struct.Struct("!I")                     # 4-byte big-endian unsigned length

def send_msg(sock, payload: bytes):
    sock.sendall(HEADER.pack(len(payload)) + payload)

def recv_exactly(sock, n: int) -> bytes:
    buf = bytearray()
    while len(buf) < n:
        chunk = sock.recv(n - len(buf))
        if not chunk:
            raise ConnectionError(f"connection closed after {len(buf)} of {n} bytes")
        buf += chunk
    return bytes(buf)

def recv_msg(sock) -> bytes:
    (length,) = HEADER.unpack(recv_exactly(sock, HEADER.size))
    return recv_exactly(sock, length)

print(HEADER.pack(5))                             # b'\x00\x00\x00\x05'

Now a server that sends three messages, deliberately dribbling the bytes out in tiny pieces. The client still gets exactly three intact messages:

In [ ]:
def chunky_sender(listener_sock):
    conn, _ = listener_sock.accept()
    with conn:
        frames = b"".join(HEADER.pack(len(m)) + m for m in [b"alpha", b"beta", "ünïcode".encode()])
        for i in range(0, len(frames), 3):        # 3 bytes at a time
            conn.sendall(frames[i:i + 3])
            time.sleep(0.005)

with socket.create_server(("127.0.0.1", 0)) as srv:
    t = threading.Thread(target=chunky_sender, args=(srv,))
    t.start()
    with socket.create_connection(srv.getsockname(), timeout=5) as c:
        messages = [recv_msg(c) for _ in range(3)]
    t.join()

print(messages)
print(messages[2].decode())

The `ConnectionError` branch matters too: if the peer disconnects mid-message, you get a clear error instead of a half message.

In [ ]:
def hang_up_early(listener_sock):
    conn, _ = listener_sock.accept()
    with conn:
        conn.sendall(HEADER.pack(100) + b"only a few bytes")   # promises 100, sends 16

with socket.create_server(("127.0.0.1", 0)) as srv:
    t = threading.Thread(target=hang_up_early, args=(srv,))
    t.start()
    with socket.create_connection(srv.getsockname(), timeout=5) as c:
        try:
            recv_msg(c)
        except ConnectionError as e:
            print(f"{type(e).__name__}: {e}")
    t.join()

## Timeouts

Sockets are **blocking with no timeout** by default — a `recv` on a silent peer waits forever. Set one with `settimeout(seconds)` (or the `timeout=` argument of `create_connection`). A timeout raises the built-in `TimeoutError` (`socket.timeout` is an alias of it).

| Call | Meaning |
|---|---|
| `s.settimeout(None)` | blocking, wait forever (default) |
| `s.settimeout(2.5)` | blocking with a timeout |
| `s.settimeout(0)` / `s.setblocking(False)` | non-blocking: raise immediately if not ready |

In [ ]:
def silent_server(listener_sock, done):
    conn, _ = listener_sock.accept()
    with conn:
        done.wait(1)                              # accept, then say nothing

with socket.create_server(("127.0.0.1", 0)) as srv:
    done = threading.Event()
    t = threading.Thread(target=silent_server, args=(srv, done))
    t.start()
    with socket.create_connection(srv.getsockname(), timeout=0.2) as c:
        start = time.perf_counter()
        try:
            c.recv(1024)
        except TimeoutError as e:
            print(f"{type(e).__name__}: {e} after ~{time.perf_counter() - start:.1f}s")
    done.set()
    t.join()

print(socket.timeout is TimeoutError)

⚠️ **Gotcha:** connecting to a port with nothing listening fails fast with `ConnectionRefusedError` (a subclass of `ConnectionError`/`OSError`); connecting to an unreachable host waits for the timeout. Always set one for client sockets.

In [ ]:
with socket.create_server(("127.0.0.1", 0)) as tmp:
    free_port = tmp.getsockname()[1]              # nothing listens there after this block

try:
    socket.create_connection(("127.0.0.1", free_port), timeout=1)
except ConnectionRefusedError as e:
    print(f"{type(e).__name__}: connection refused")

## Stopping the echo server

Our accept loop checks the `stop` event every 0.1 s (thanks to the listener's timeout). Set it, wait a moment for the loop to exit, then close the listening socket:

In [ ]:
stop.set()
time.sleep(0.2)
listener.close()
print("echo server stopped")

## UDP in brief

```csharp
// C#
using var udp = new UdpClient(new IPEndPoint(IPAddress.Loopback, 0));
await udp.SendAsync(data, data.Length, remoteEndPoint);
var result = await udp.ReceiveAsync();
```

UDP (`SOCK_DGRAM`) is connectionless and **message-oriented**: each `sendto` is one datagram, received whole by one `recvfrom` (no framing problem) — but delivery and ordering are **not guaranteed**, and a datagram larger than your buffer is truncated. Used for DNS, metrics (StatsD), discovery, games.

In [ ]:
with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as receiver, \
     socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as sender:
    receiver.bind(("127.0.0.1", 0))
    receiver.settimeout(2)

    sender.sendto(b"cpu=0.42", receiver.getsockname())
    sender.sendto(b"mem=0.73", receiver.getsockname())

    for _ in range(2):
        data, addr = receiver.recvfrom(1024)
        print(data, "from 127.0.0.1:", addr[0] == "127.0.0.1")

## `socketserver` — less boilerplate

The `socketserver` module packages the accept loop for you (it is what `http.server` is built on). Subclass `StreamRequestHandler`, implement `handle()`, and use `self.rfile`/`self.wfile` as file objects — line-based framing comes for free with `readline()`:

In [ ]:
import socketserver

class LineEcho(socketserver.StreamRequestHandler):
    def handle(self):
        for line in self.rfile:                   # one request line at a time
            self.wfile.write(line.upper())

with socketserver.ThreadingTCPServer(("127.0.0.1", 0), LineEcho) as ss:
    ss.daemon_threads = True
    t = threading.Thread(target=ss.serve_forever, daemon=True)
    t.start()
    with socket.create_connection(ss.server_address, timeout=5) as c, c.makefile("rwb") as f:
        for word in [b"one\n", b"two\n"]:
            f.write(word)
            f.flush()
            print(f.readline())
    ss.shutdown()

**Tip:** for new network code that has to handle many connections, prefer asyncio streams (`asyncio.start_server` / `open_connection`) *(see Standard Library › Concurrency › 005 · asyncio in Depth)* — one thread, thousands of connections, and `readline()`/`readexactly()` built in.

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Listen | `new TcpListener(ep).Start()` | `socket.create_server((host, port))` |
| Accept | `AcceptTcpClient()` | `conn, addr = srv.accept()` |
| Connect | `new TcpClient(host, port)` | `socket.create_connection((host, port), timeout=t)` |
| Send all | `stream.Write(bytes)` | `sock.sendall(data)` |
| Receive | `stream.Read(buffer)` | `sock.recv(n)` (may be partial; `b""` = closed) |
| Read exactly n bytes | `stream.ReadExactly(buffer)` (.NET 7+) | loop over `recv` (`recv_exactly` helper) |
| Length prefix | `BinaryPrimitives.WriteUInt32BigEndian` | `struct.pack("!I", n)` |
| Timeout | `ReceiveTimeout` | `sock.settimeout(sec)` → `TimeoutError` |
| Dispose | `using` | `with socket... as s:` |
| UDP | `UdpClient` | `SOCK_DGRAM` + `sendto` / `recvfrom` |
| Server framework | — | `socketserver.ThreadingTCPServer` |

## Exercises

1. Write a TCP server (port 0, background thread) that receives one length-prefixed message containing a UTF-8 string and replies with a length-prefixed message containing the string reversed. Test it with `"stressed"`.
2. Write `recv_line(sock)` that reads byte by byte until `b"\n"` and returns the line without the newline. Test it against a server that sends `b"HELLO\nWORLD\n"` in one `sendall`, calling `recv_line` twice.
3. Use two UDP sockets to send the three datagrams `b"a"`, `b"bb"`, `b"ccc"` and print the length of each received datagram.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import socket, struct, threading

HEADER = struct.Struct("!I")

def recv_exactly(sock, n):
    buf = bytearray()
    while len(buf) < n:
        chunk = sock.recv(n - len(buf))
        if not chunk:
            raise ConnectionError("closed early")
        buf += chunk
    return bytes(buf)

def send_msg(sock, payload):
    sock.sendall(HEADER.pack(len(payload)) + payload)

def recv_msg(sock):
    (n,) = HEADER.unpack(recv_exactly(sock, HEADER.size))
    return recv_exactly(sock, n)

# 1 — length-prefixed request/response
def reverse_server(srv):
    conn, _ = srv.accept()
    with conn:
        text = recv_msg(conn).decode()
        send_msg(conn, text[::-1].encode())

with socket.create_server(("127.0.0.1", 0)) as srv:
    t = threading.Thread(target=reverse_server, args=(srv,))
    t.start()
    with socket.create_connection(srv.getsockname(), timeout=5) as c:
        send_msg(c, "stressed".encode())
        print(recv_msg(c).decode())          # desserts
    t.join()

# 2 — delimiter framing
def recv_line(sock):
    buf = bytearray()
    while (b := sock.recv(1)) not in (b"", b"\n"):
        buf += b
    return bytes(buf)

def lines_server(srv):
    conn, _ = srv.accept()
    with conn:
        conn.sendall(b"HELLO\nWORLD\n")

with socket.create_server(("127.0.0.1", 0)) as srv:
    t = threading.Thread(target=lines_server, args=(srv,))
    t.start()
    with socket.create_connection(srv.getsockname(), timeout=5) as c:
        print(recv_line(c), recv_line(c))
    t.join()

# 3 — UDP datagrams keep their boundaries
with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as rx, \
     socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as tx:
    rx.bind(("127.0.0.1", 0))
    rx.settimeout(2)
    for d in [b"a", b"bb", b"ccc"]:
        tx.sendto(d, rx.getsockname())
    print([len(rx.recvfrom(1024)[0]) for _ in range(3)])
```

</details>